In [ ]:
!pip install --upgrade torch torchvision diffusers accelerate safetensors matplotlib pillow bitsandbytes

In [ ]:
import torch
import json
import gradio as gr
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline, DDIMScheduler
from safetensors.torch import load_file
from PIL import Image, ImageFilter, ImageEnhance

# --- MODEL YÜKLEME BÖLÜMÜ ---
yapilandirma_yolu = "/content/drive/MyDrive/Cidik/Image_production/botanical_diffusion/botanical_final.json"
model_agirlik_yolu = "/content/drive/MyDrive/Cidik/Image_production/botanical_diffusion/final_botanic_model.safetensors"
cihaz = "cuda" if torch.cuda.is_available() else "cpu"

def modeli_hazirla():
    with open(yapilandirma_yolu, "r") as f:
        config = json.load(f)
    unet = UNet2DModel.from_config(config).to(cihaz)
    agirliklar = load_file(model_agirlik_yolu, device=cihaz)
    unet.load_state_dict(agirliklar)
    unet.eval()

    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2", prediction_type="epsilon")
    pipe = DDPMPipeline(unet=unet, scheduler=scheduler).to(cihaz)
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    return pipe

pipe = modeli_hazirla()

# --- ÜRETİM FONKSİYONU ---
def generate(seed, steps, sharpness, contrast):
    generator = torch.manual_seed(int(seed))

    with torch.no_grad():
        image = pipe(num_inference_steps=int(steps), generator=generator).images[0]

    # Post-Processing
    image = image.resize((512, 512), resample=Image.LANCZOS)
    for _ in range(int(sharpness)):
        image = image.filter(ImageFilter.SHARPEN)

    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(contrast)

    return image

# --- GRADIO ARAYÜZÜ (UI) ---
with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald")) as demo:
    gr.Markdown("""
    # Botanical Diffusion Pro
    **Tesla T4 Optimize Edilmiş Yüksek Kaliteli Çiçek Üretim Arayüzü**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            seed_input = gr.Number(label="Seed (Rastgelelik)", value=42)
            steps_slider = gr.Slider(minimum=10, maximum=200, step=1, label="Üretim Adımı (Steps)", value=100)
            sharp_slider = gr.Slider(minimum=0, maximum=5, step=1, label="Keskinlik Seviyesi", value=2)
            contrast_slider = gr.Slider(minimum=0.5, maximum=2.0, step=0.1, label="Kontrast Oranı", value=1.3)
            btn = gr.Button("Görsel Üret", variant="primary")

        with gr.Column(scale=2):
            output_img = gr.Image(label="Üretilen Botanik Görsel", type="pil", interactive=False)

    btn.click(fn=generate, inputs=[seed_input, steps_slider, sharp_slider, contrast_slider], outputs=output_img)

    gr.Examples(
        examples=[[123, 120, 2, 1.3], [456, 150, 3, 1.4], [789, 100, 1, 1.2]],
        inputs=[seed_input, steps_slider, sharp_slider, contrast_slider]
    )

if __name__ == "__main__":
    demo.launch(share=True) # share=True sayesinde dışarıdan erişilebilir link verir

/tmp/ipykernel_180/3116182052.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4db65d0f57d195eb0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
